# Notebook 12 — Robustness, Stress Testing & Final Model Freeze

This notebook is the final research-validation stage before production/deployment.

It evaluates whether the walk-forward strategy remains robust when changing:
- probability threshold
- transaction costs
- slippage
- calendar year
- volatility regime

It then freezes a research configuration before the final untouched evaluation.

**Important:** do not tune the frozen configuration after observing the final holdout.


In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    for c in [Path.cwd(), Path.cwd().parent, Path("/mnt/data/quant-trading-research")]:
        if (c / "data").exists() and (c / "notebooks").exists():
            ROOT = c
            break

MASTER_PATH = ROOT / "data" / "raw" / "sp500_1950_present.csv"
WF_PATH = ROOT / "data" / "interim" / "sp500_walk_forward_predictions.parquet"
INTERIM = ROOT / "data" / "interim"
FIGURES = ROOT / "reports" / "figures"
TABLES = ROOT / "reports" / "tables"
REPORTS = ROOT / "reports" / "generated"
MODEL_DIR = ROOT / "models" / "frozen"

for x in [INTERIM, FIGURES, TABLES, REPORTS, MODEL_DIR]:
    x.mkdir(parents=True, exist_ok=True)

TRADING_DAYS = 252

print("Project root:", ROOT)


In [ ]:
if not WF_PATH.exists():
    raise FileNotFoundError(
        f"{WF_PATH} not found. Run Notebook 10 first."
    )

wf = pd.read_parquet(WF_PATH)
wf["Date"] = pd.to_datetime(wf["Date"], errors="coerce")
wf = wf.sort_values(["model", "Date"]).reset_index(drop=True)

required = [
    "Date", "Close", "next_day_return", "target",
    "model", "probability_up", "prediction", "fold"
]
missing = [c for c in required if c not in wf.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

assert wf["Date"].notna().all()
assert wf["probability_up"].between(0, 1).all()

print("Rows:", len(wf))
print("Models:", sorted(wf["model"].unique()))
print("Range:", wf["Date"].min().date(), "to", wf["Date"].max().date())


In [ ]:
def summarize(returns, position, name):
    returns = pd.Series(returns).fillna(0.0)
    position = pd.Series(position, index=returns.index).fillna(0.0)

    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1
    years = len(equity) / TRADING_DAYS
    cagr = equity.iloc[-1] ** (1 / years) - 1 if years > 0 and equity.iloc[-1] > 0 else np.nan

    vol = returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe = returns.mean() / returns.std(ddof=1) * np.sqrt(TRADING_DAYS) if returns.std(ddof=1) > 0 else np.nan

    downside = returns[returns < 0]
    if len(downside):
        ddv = np.sqrt(np.mean(downside ** 2))
        sortino = returns.mean() / ddv * np.sqrt(TRADING_DAYS) if ddv > 0 else np.nan
    else:
        sortino = np.inf

    turnover = position.diff().abs().fillna(position.abs())

    summary = {
        "strategy": name,
        "cumulative_return": equity.iloc[-1] - 1,
        "CAGR": cagr,
        "annualized_volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": drawdown.min(),
        "Calmar": cagr / abs(drawdown.min()) if drawdown.min() < 0 else np.nan,
        "win_rate": (returns[position != 0] > 0).mean() if (position != 0).any() else np.nan,
        "trades": int((turnover > 0).sum()),
        "turnover": turnover.sum(),
        "average_position": position.mean(),
    }
    return summary, equity, drawdown


def run_strategy(frame, threshold=0.50, cost_bps=5.0, slippage_bps=2.0):
    frame = frame.sort_values("Date").reset_index(drop=True).copy()

    position = (frame["probability_up"] >= threshold).astype(float)
    gross = position * frame["next_day_return"]

    turnover = position.diff().abs().fillna(position.abs())
    cost_rate = (cost_bps + slippage_bps) / 10000
    net = gross - turnover * cost_rate

    result = frame[
        ["Date", "Close", "next_day_return",
         "probability_up", "target", "fold"]
    ].copy()

    result["position"] = position
    result["turnover"] = turnover
    result["cost"] = turnover * cost_rate
    result["gross_return"] = gross
    result["net_return"] = net
    result["equity"] = (1 + net).cumprod()

    summary, equity, drawdown = summarize(
        net, position, f"Threshold {threshold:.2f}"
    )
    return result, summary, equity, drawdown


In [ ]:
selection_path = TABLES / "sp500_walk_forward_selection_table.csv"

if selection_path.exists():
    selection = pd.read_csv(selection_path)
    candidate_model = selection.iloc[0]["model"]
else:
    from sklearn.metrics import roc_auc_score
    pooled = []
    for model in wf["model"].unique():
        s = wf[wf["model"] == model]
        pooled.append({
            "model": model,
            "roc_auc": roc_auc_score(
                s["target"], s["probability_up"]
            )
        })
    candidate_model = pd.DataFrame(pooled).sort_values(
        "roc_auc", ascending=False
    ).iloc[0]["model"]

candidate = wf[
    wf["model"] == candidate_model
].sort_values("Date").reset_index(drop=True)

print("Frozen candidate model to evaluate:", candidate_model)
print("Observations:", len(candidate))


In [ ]:
thresholds = [0.50, 0.52, 0.55, 0.57, 0.60, 0.62, 0.65, 0.70]
rows = []

for threshold in thresholds:
    _, s, _, _ = run_strategy(
        candidate,
        threshold=threshold,
        cost_bps=5,
        slippage_bps=2
    )
    rows.append({
        "threshold": threshold,
        "CAGR": s["CAGR"],
        "Sharpe": s["Sharpe"],
        "Sortino": s["Sortino"],
        "max_drawdown": s["max_drawdown"],
        "cumulative_return": s["cumulative_return"],
        "win_rate": s["win_rate"],
        "trades": s["trades"],
        "turnover": s["turnover"],
        "average_position": s["average_position"],
    })

threshold_df = pd.DataFrame(rows)
display(threshold_df)
threshold_df.to_csv(
    TABLES / "sp500_robustness_threshold_analysis.csv",
    index=False
)


In [ ]:
fig = plt.figure(figsize=(11, 6))
plt.plot(threshold_df["threshold"], threshold_df["Sharpe"], marker="o")
plt.axhline(0, linestyle="--")
plt.title(f"Threshold Robustness — {candidate_model}")
plt.xlabel("Probability Threshold")
plt.ylabel("Sharpe Ratio")
plt.grid(True, alpha=0.25)
plt.tight_layout()
path = FIGURES / "sp500_robustness_threshold_sharpe.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)


In [ ]:
cost_levels = [0, 2, 5, 10, 15, 20, 30]
slippage_levels = [0, 2, 5, 10]
rows = []

for cost in cost_levels:
    for slip in slippage_levels:
        _, s, _, _ = run_strategy(
            candidate,
            threshold=0.50,
            cost_bps=cost,
            slippage_bps=slip
        )
        rows.append({
            "transaction_cost_bps": cost,
            "slippage_bps": slip,
            "total_cost_bps": cost + slip,
            "CAGR": s["CAGR"],
            "Sharpe": s["Sharpe"],
            "Sortino": s["Sortino"],
            "max_drawdown": s["max_drawdown"],
            "cumulative_return": s["cumulative_return"],
            "trades": s["trades"],
            "turnover": s["turnover"],
        })

stress_df = pd.DataFrame(rows)
display(stress_df)
stress_df.to_csv(
    TABLES / "sp500_robustness_cost_slippage_stress.csv",
    index=False
)


In [ ]:
heatmap = stress_df.pivot(
    index="transaction_cost_bps",
    columns="slippage_bps",
    values="Sharpe"
)
display(heatmap)

fig = plt.figure(figsize=(9, 6))
plt.imshow(heatmap.values, aspect="auto")
plt.xticks(range(len(heatmap.columns)), heatmap.columns)
plt.yticks(range(len(heatmap.index)), heatmap.index)
plt.colorbar(label="Sharpe")
plt.title("Sharpe Under Cost + Slippage Stress")
plt.xlabel("Slippage (bps)")
plt.ylabel("Transaction Cost (bps)")
plt.tight_layout()
path = FIGURES / "sp500_robustness_cost_heatmap.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)


In [ ]:
base_frame, base_summary, base_equity, base_dd = run_strategy(
    candidate,
    threshold=0.50,
    cost_bps=5,
    slippage_bps=2
)

base_frame["year"] = base_frame["Date"].dt.year

rows = []
for year, g in base_frame.groupby("year"):
    s, equity, dd = summarize(
        g["net_return"],
        g["position"],
        str(year)
    )
    rows.append({
        "year": year,
        "return": equity.iloc[-1] - 1,
        "CAGR": s["CAGR"],
        "Sharpe": s["Sharpe"],
        "Sortino": s["Sortino"],
        "max_drawdown": s["max_drawdown"],
        "trades": s["trades"],
        "average_position": s["average_position"],
    })

annual_df = pd.DataFrame(rows)
display(annual_df)
annual_df.to_csv(
    TABLES / "sp500_robustness_annual_performance.csv",
    index=False
)


In [ ]:
fig = plt.figure(figsize=(14, 6))
plt.bar(annual_df["year"].astype(str), annual_df["return"])
plt.axhline(0, linestyle="--")
plt.title(f"Annual Returns — {candidate_model}")
plt.xlabel("Year")
plt.ylabel("Net Return")
plt.xticks(rotation=45)
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
path = FIGURES / "sp500_robustness_annual_returns.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)


In [ ]:
regime = candidate.copy()
regime["daily_return"] = regime["Close"].pct_change()
regime["rolling_volatility"] = (
    regime["daily_return"].rolling(21).std() * np.sqrt(TRADING_DAYS)
)
median_vol = regime["rolling_volatility"].median()
regime["volatility_regime"] = np.where(
    regime["rolling_volatility"] <= median_vol,
    "Low Volatility",
    "High Volatility"
)

regime_frame = base_frame.copy()
regime_frame["rolling_volatility"] = regime["rolling_volatility"].values
regime_frame["volatility_regime"] = regime["volatility_regime"].values

rows = []
for name, g in regime_frame.dropna(
    subset=["rolling_volatility"]
).groupby("volatility_regime"):
    s, equity, dd = summarize(
        g["net_return"],
        g["position"],
        name
    )
    rows.append({
        "regime": name,
        "observations": len(g),
        "return": equity.iloc[-1] - 1,
        "CAGR": s["CAGR"],
        "Sharpe": s["Sharpe"],
        "Sortino": s["Sortino"],
        "max_drawdown": s["max_drawdown"],
        "average_position": s["average_position"],
    })

regime_df = pd.DataFrame(rows)
display(regime_df)
regime_df.to_csv(
    TABLES / "sp500_robustness_volatility_regimes.csv",
    index=False
)


In [ ]:
cal = candidate.copy()
cal["probability_bin"] = pd.cut(
    cal["probability_up"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

calibration_df = (
    cal.groupby("probability_bin", observed=False)
    .agg(
        observations=("target", "size"),
        mean_predicted_probability=("probability_up", "mean"),
        actual_positive_rate=("target", "mean")
    )
    .reset_index()
)

display(calibration_df)
calibration_df.to_csv(
    TABLES / "sp500_robustness_probability_calibration.csv",
    index=False
)


In [ ]:
valid = calibration_df[calibration_df["observations"] > 0]

fig = plt.figure(figsize=(8, 8))
plt.plot(
    valid["mean_predicted_probability"],
    valid["actual_positive_rate"],
    marker="o",
    label="Model"
)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.title(f"Probability Calibration — {candidate_model}")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Actual Positive Rate")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
path = FIGURES / "sp500_robustness_probability_calibration.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)


In [ ]:
always_position = pd.Series(1.0, index=candidate.index)
always_summary, always_equity, always_dd = summarize(
    candidate["next_day_return"].fillna(0),
    always_position,
    "Always Long"
)

baseline_df = pd.DataFrame([
    {"strategy": "Always Long", **always_summary},
    {"strategy": "Candidate Long/Cash", **base_summary},
])

display(baseline_df)
baseline_df.to_csv(
    TABLES / "sp500_robustness_always_long_comparison.csv",
    index=False
)


In [ ]:
rng = np.random.default_rng(42)
daily = base_frame["net_return"].dropna().to_numpy()
boot = []

for _ in range(2000):
    sample = rng.choice(daily, size=len(daily), replace=True)
    std = sample.std(ddof=1)
    if std > 0:
        boot.append(
            sample.mean() / std * np.sqrt(TRADING_DAYS)
        )

boot = np.asarray(boot)
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

bootstrap_df = pd.DataFrame([{
    "observed_sharpe": base_summary["Sharpe"],
    "bootstrap_mean_sharpe": boot.mean(),
    "bootstrap_2_5_percentile": ci_low,
    "bootstrap_97_5_percentile": ci_high,
    "bootstrap_samples": len(boot)
}])

display(bootstrap_df)
bootstrap_df.to_csv(
    TABLES / "sp500_robustness_bootstrap_sharpe.csv",
    index=False
)


In [ ]:
scorecard = pd.DataFrame([{
    "candidate_model": candidate_model,
    "baseline_sharpe": base_summary["Sharpe"],
    "positive_threshold_fraction": (threshold_df["Sharpe"] > 0).mean(),
    "positive_cost_stress_fraction": (stress_df["Sharpe"] > 0).mean(),
    "positive_year_fraction": (annual_df["return"] > 0).mean(),
    "positive_regime_fraction": (regime_df["Sharpe"] > 0).mean(),
    "bootstrap_sharpe_ci_low": ci_low,
    "bootstrap_sharpe_ci_high": ci_high,
}])

display(scorecard)
scorecard.to_csv(
    TABLES / "sp500_robustness_scorecard.csv",
    index=False
)


In [ ]:
FROZEN_THRESHOLD = 0.50
FROZEN_TRANSACTION_COST_BPS = 5.0
FROZEN_SLIPPAGE_BPS = 2.0

frozen_config = {
    "status": "FROZEN_RESEARCH_CONFIGURATION",
    "candidate_model": candidate_model,
    "signal_type": "LONG_CASH",
    "probability_threshold": FROZEN_THRESHOLD,
    "transaction_cost_bps": FROZEN_TRANSACTION_COST_BPS,
    "slippage_bps": FROZEN_SLIPPAGE_BPS,
    "total_cost_bps": (
        FROZEN_TRANSACTION_COST_BPS +
        FROZEN_SLIPPAGE_BPS
    ),
    "walk_forward_validation": True,
    "parameter_optimization_after_freeze": False,
    "do_not_tune_after_final_holdout": True
}

config_path = MODEL_DIR / "sp500_frozen_research_config.json"
config_path.write_text(
    json.dumps(frozen_config, indent=2),
    encoding="utf-8"
)

print(json.dumps(frozen_config, indent=2))
print("Saved:", config_path)


In [ ]:
frozen_frame, frozen_summary, frozen_equity, frozen_dd = run_strategy(
    candidate,
    threshold=FROZEN_THRESHOLD,
    cost_bps=FROZEN_TRANSACTION_COST_BPS,
    slippage_bps=FROZEN_SLIPPAGE_BPS
)

frozen_path = INTERIM / "sp500_frozen_strategy_backtest.parquet"
frozen_frame.to_parquet(frozen_path, index=False)

metadata = {
    "candidate_model": candidate_model,
    "prediction_source": str(WF_PATH),
    "target": "next_day_return > 0",
    "threshold": FROZEN_THRESHOLD,
    "transaction_cost_bps": FROZEN_TRANSACTION_COST_BPS,
    "slippage_bps": FROZEN_SLIPPAGE_BPS,
}

metadata_path = MODEL_DIR / "sp500_frozen_model_metadata.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

display(pd.DataFrame([frozen_summary]))
print("Saved:", frozen_path)
print("Saved:", metadata_path)


In [ ]:
report = {
    "candidate_model": candidate_model,
    "frozen_configuration": frozen_config,
    "frozen_performance": frozen_summary,
    "threshold_analysis": threshold_df.to_dict(orient="records"),
    "cost_slippage_stress": stress_df.to_dict(orient="records"),
    "annual_performance": annual_df.to_dict(orient="records"),
    "volatility_regimes": regime_df.to_dict(orient="records"),
    "calibration": calibration_df.to_dict(orient="records"),
    "bootstrap_sharpe": bootstrap_df.to_dict(orient="records"),
    "robustness_scorecard": scorecard.to_dict(orient="records"),
    "methodological_note": (
        "The research configuration is frozen before the final untouched "
        "evaluation. Final holdout results must not be used to tune model, "
        "threshold, features, hyperparameters, or cost assumptions."
    )
}

report_path = REPORTS / "sp500_robustness_stress_testing_report.json"
report_path.write_text(
    json.dumps(report, indent=2, default=str),
    encoding="utf-8"
)

print(json.dumps(report, indent=2, default=str))
print("Saved:", report_path)


In [ ]:
master = pd.read_csv(MASTER_PATH, low_memory=False)
expected = ["Date","Open","High","Low","Close","Adj.Close","Volume"]
assert list(master.columns) == expected
dates = pd.to_datetime(master["Date"], errors="coerce")
assert dates.notna().all()
assert dates.is_unique
assert dates.is_monotonic_increasing

print("Raw master dataset integrity after robustness testing: PASS")
print("Master rows:", len(master))


# Notebook 12 Complete

Completed:
- threshold robustness
- transaction-cost stress testing
- slippage stress testing
- annual performance analysis
- volatility-regime analysis
- probability calibration
- Always-Long comparison
- bootstrap Sharpe interval
- robustness scorecard
- frozen research configuration
- frozen strategy artifact
- frozen metadata
- JSON research report
- final raw-data integrity check

**Next:** Notebook 13 — Final Untouched Test, Statistical Significance & Research Conclusions.

Run Notebook 12 completely and verify the results before continuing.
